In [0]:
# Parametro do formato do nome da tabela gerada ao final. Ex: 202404
dbutils.widgets.text("nmmes", "")

In [0]:
%sql

SELECT * FROM databox.juridico_comum.tb_fecham_financ_civel_202412

In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

In [0]:
from pyspark.sql.functions import col

nmmes = dbutils.widgets.get("nmmes")
nmmes = '202411'

df_base_tabelao_civel = spark.sql(f'''
   select *
    ,CASE WHEN CADASTRO <= '2020-01-01' THEN 'NOVO' ELSE 'LEGADO' END AS NOVO_X_LEGADO
   from databox.juridico_comum.tb_fecham_financ_civel_{nmmes}                   
''')

df_base_tabelao_civel = df_base_tabelao_civel.withColumn('NOVOS', col('NOVOS').cast('double'))

df_base_tabelao_civel = df_base_tabelao_civel.fillna(0)

df_base_tabelao_civel.createOrReplaceTempView('df_base_tabelao_civel')

In [0]:
display(df_base_tabelao_civel)

In [0]:
df_dados_1 = spark.sql('''

select SUM(ESTOQUE) AS ESTOQUE_QTD,
 sum(coalesce(PROVISAO_TOTAL_M,0)) / 1000 AS SALDO_PROVISAO,
--  sum(coalesce(CORRECAO_M,0)) / 1000  AS MOVIMENTACAO_CORRECAO,
--  sum(coalesce(CORRECAO_M_1,0)) / 1000  AS SALDO_CORRECAO,
 sum(coalesce(PROVISAO_TOTAL_PASSIVO_M, 0)) / 1000  as SALDO_PROVISAO_TOTAL, 
 (sum(coalesce(PROVISAO_TOTAL_PASSIVO_M, 0)) / count(estoque)) / 1000  AS TKM_MEDIO,
 SUM(NOVOS) AS ENTRADAS_QTD,
 SUM(CASE WHEN NOVOS = 1 THEN coalesce(PROVISAO_TOTAL_PASSIVO_M, 0) ELSE 0 END)  / 1000  AS ENTRADAS_VLR,
 (SUM(CASE WHEN NOVOS = 1 THEN coalesce(PROVISAO_TOTAL_PASSIVO_M, 0) ELSE 0 END) / COUNT(NOVOS)) / 1000  AS TKM_ENTRADAS,
 SUM(ENCERRADOS) AS ENCERRADOS_QTD,
 SUM(CASE WHEN ENCERRADOS = 1 THEN COALESCE(TOTAL_PAGAMENTOS, 0) ELSE 0 END) / 1000  AS TOTAL_PAGAMENTOS,
 (SUM(CASE WHEN ENCERRADOS = 1 THEN COALESCE(TOTAL_PAGAMENTOS, 0) ELSE 0 END) / count(ENCERRADOS)) / 1000  AS TKM_MEDIO_ENCERRADOS,
 COUNT(CASE WHEN ESTOQUE = 1 AND coalesce(PROVISAO_TOTAL_PASSIVO_M, 0) <> 0 THEN 1 END) AS ESTOQUE_COM_PROVISAO,
 (SUM(CASE WHEN ESTOQUE = 1 THEN COALESCE(PROVISAO_TOTAL_PASSIVO_M, 0) ELSE 0 END) / COUNT(ESTOQUE)) / 1000  AS TKM_MEDIO_COM_PROV,
 (SUM(CASE WHEN ENCERRADOS = 1 AND NOVO_X_LEGADO = 'NOVO' THEN COALESCE(TOTAL_PAGAMENTOS, 0) ELSE 0 END) / COUNT(CASE WHEN NOVO_X_LEGADO = 'NOVO' AND ENCERRADOS = 1 THEN 1 END)) /1000 AS TKM_MEDIO_NOVO,
 (SUM(CASE WHEN ENCERRADOS = 1 AND NOVO_X_LEGADO = 'LEGADO' THEN COALESCE(TOTAL_PAGAMENTOS, 0) ELSE 0 END) / COUNT(CASE WHEN NOVO_X_LEGADO = 'LEGADO' AND ENCERRADOS = 1 THEN 1 END)) /1000 AS TKM_MEDIO_LEGADO,
 mes_fech 
 from df_base_tabelao_civel 
group by mes_fech
''')

df_dados_2 = spark.sql('''
    select COUNT(ENCERRADOS) AS SEM_ONUS_QTD, mes_fech from df_base_tabelao_civel
    where MOTIVO_ENC_AGRP = 'SEM ONUS'
    GROUP BY mes_fech
''')

df_dados_3 = spark.sql('''
    select COUNT(ENCERRADOS) AS CONDENACAO_ACORDO_QTD, mes_fech from df_base_tabelao_civel
    where MOTIVO_ENC_AGRP <> 'SEM ONUS'
    GROUP BY mes_fech
''')


df_dados_4 = spark.sql('''
    select COUNT(ENCERRADOS) AS CONDENACAO_QTD, mes_fech from df_base_tabelao_civel
    where MOTIVO_ENC_AGRP = 'CONDENACAO'
    GROUP BY mes_fech
''')

df_dados_5 = spark.sql('''
    select COUNT(ENCERRADOS) AS ACORDO_QTD, mes_fech from df_base_tabelao_civel
    where MOTIVO_ENC_AGRP = 'ACORDO'
    GROUP BY mes_fech
''')

df_dados_1.createOrReplaceTempView('df_dados_1')
df_dados_2.createOrReplaceTempView('df_dados_2')
df_dados_3.createOrReplaceTempView('df_dados_3') 
df_dados_4.createOrReplaceTempView('df_dados_4')
df_dados_5.createOrReplaceTempView('df_dados_5')


In [0]:
df_tabelao_civel = spark.sql('''

select A.*,
    B.SEM_ONUS_QTD,
    C.CONDENACAO_ACORDO_QTD,
    D.CONDENACAO_QTD,
    E.ACORDO_QTD

from df_dados_1 A 
left join df_dados_2 B
ON A.mes_fech = B.mes_fech
left join df_dados_3 C
ON A.mes_fech = C.mes_fech
left join df_dados_4 D
ON A.mes_fech = D.mes_fech
left join df_dados_5 E
ON A.mes_fech = E.mes_fech
''')



In [0]:
display(df_tabelao_civel)

In [0]:
import pandas as pd
import re
from shutil import copyfile
import os

# Converter PySpark DataFrame para Pandas DataFrame
try:
    pandas_df = df_tabelao_civel.toPandas()
except:
    pass

pandas_df = df_tabelao_civel.toPandas()

# Salvar o DataFrame com os nomes corrigidos em um arquivo Excel no disco local primeiro
local_path = '/local_disk0/tmp/df_tabelao_civel.xlsx'
pandas_df.to_excel(local_path, index=False, sheet_name='df_tabelao_civel')

# Definir o nome do arquivo
qtd = 0
path = '/Volumes/databox/juridico_comum/arquivos/modelo_provisao/output/'
list_files = dbutils.fs.ls(path)
nome_arquivo = f'df_tabelao_civel'

# Essa rotina verifica quantos arquivos com o nome escolhido existem na pasta e faz um controle de versao com base na quantidade
for file in list_files:
    name = file.name
    if nome_arquivo in name:
        qtd += 1
        print(f'{name} | {qtd} \n')

if qtd == 0:
    print(f'Ainda não foi criado nenhum arquivo com o nome: {nome_arquivo}')
if qtd >= 1:
    new_qtd = qtd +1
    nome_arquivo = f'df_tabelao_civel_{new_qtd}'


# Copiar o arquivo do disco local para o volume desejado
volume_path = f'/Volumes/databox/juridico_comum/arquivos/modelo_provisao/output/{nome_arquivo}.xlsx'
copyfile(local_path, volume_path)